In [22]:
import os
import sys

# --- RIFE source / model ---
RIFE_ROOT   = "/kaggle/input/datasets/jaisai002/rife-kaggle/RIFE_kaggle"

# --- Data ---
VIMEO_ROOT      = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
SPLITS_PATH     = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"
CLASSIFIED_CSV  = "/kaggle/input/datasets/hibro00/vfi-csvs/raft_descriptors_train_classified.csv"
TEST_CLASSIFIED = "/kaggle/input/datasets/hibro00/vfi-csvs/test_motion_classifications.csv"

# --- Working dirs / outputs ---
OUTPUT_DIR = "/kaggle/working/rife_cnn"
BACKUP_DIR = "/kaggle/input/datasets/hibro00/vfi-pths-backup"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if RIFE_ROOT not in sys.path:
    sys.path.insert(0, RIFE_ROOT)

In [1]:
import os

base = "/kaggle/input/datasets"

for owner in os.listdir(base):
    print("\n", owner)
    path = os.path.join(base, owner)
    if os.path.isdir(path):
        for item in os.listdir(path):
            print("  ", item)


 chenshu123
   vimeo-triplet

 jaisai002
   ifrnet-kaggle
   film-kaggle
   rife-kaggle

 hibro00
   vfi-csvs
   vimeo-triplet-split
   vfi-pths-backup


In [2]:
import os
import pandas as pd

SPLIT_ROOT = "/kaggle/input/datasets/hibro00/vimeo-triplet-split"

print("Split dataset contents:")
print(os.listdir(SPLIT_ROOT))

# Find the CSV automatically
csv_files = [
    f for f in os.listdir(SPLIT_ROOT)
    if f.lower().endswith(".csv")
]

print("\nCSV files:", csv_files)

assert len(csv_files) >= 1, "No CSV found in split dataset."

SPLITS_PATH = os.path.join(SPLIT_ROOT, csv_files[0])
splits = pd.read_csv(SPLITS_PATH)

print("\nUsing:", SPLITS_PATH)
print("Shape:", splits.shape)
print("\nColumns:", list(splits.columns))
print("\nSplit counts:")
print(splits["split"].value_counts())

print("\nFirst rows:")
display(splits.head())

Split dataset contents:
['splits.csv']

CSV files: ['splits.csv']

Using: /kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv
Shape: (55095, 2)

Columns: ['clip_id', 'split']

Split counts:
split
train          51324
test            3601
calibration      170
Name: count, dtype: int64

First rows:


,clip_id,split
0,00001/0001,train
1,00001/0002,train
2,00001/0003,train
3,00001/0004,train
4,00001/0005,train


In [3]:
# Validate the expected split structure

calib = splits[splits["split"] == "calibration"]
test = splits[splits["split"] == "test"]

print("Calibration:", len(calib))
print("Test:", len(test))

overlap = set(calib["clip_id"]) & set(test["clip_id"])

print("Calibration/Test overlap:", len(overlap))

assert len(calib) == 170, f"Expected 170 calibration clips, got {len(calib)}"
assert len(test) == 3601, f"Expected 3601 test clips, got {len(test)}"
assert len(overlap) == 0, "Calibration/test leakage detected!"

print("\nSPLIT VALIDATION PASSED")

Calibration: 170
Test: 3601
Calibration/Test overlap: 0

SPLIT VALIDATION PASSED


In [23]:
import shutil

backup_files = [
    "rife_generic_correction.pth",
    "rife_generic_correction_checkpoint.pth",
    "rife_correction_rotation.pth",
    "rife_correction_rotation_checkpoint.pth",
]

for fname in backup_files:
    src = os.path.join(BACKUP_DIR, fname)
    dst = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print("Restored:", fname)
    else:
        print("No backup found for:", fname)

print("\nCurrent contents of", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Restored: rife_generic_correction.pth
Restored: rife_generic_correction_checkpoint.pth
Restored: rife_correction_rotation.pth
Restored: rife_correction_rotation_checkpoint.pth

Current contents of /kaggle/working/rife_cnn
['rife_generic_correction.pth', 'rife_correction_rotation_checkpoint.pth', 'rife_generic_correction_checkpoint.pth', 'rife_correction_rotation.pth']


# ***RIFE*** : Setup Testing


In [4]:
import sys
import os
import torch
import cv2
import numpy as np

VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
RIFE_ROOT = "/kaggle/input/datasets/jaisai002/rife-kaggle/RIFE_kaggle"

clip = "00058/0745"
clip_dir = os.path.join(VIMEO_ROOT, "sequences", clip)

# Put RIFE package on Python path
if RIFE_ROOT not in sys.path:
    sys.path.insert(0, RIFE_ROOT)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("RIFE root:", RIFE_ROOT)

# Import RIFE
from model.RIFE import Model

# Load model
model = Model()
model.load_model(os.path.join(RIFE_ROOT, "train_log"), -1)
model.eval()
model.device()

print("RIFE model loaded successfully.")

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    tensor = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0)
    return tensor.cuda()

# Vimeo triplet
img0 = load_image(os.path.join(clip_dir, "im1.png"))
gt   = load_image(os.path.join(clip_dir, "im2.png"))
img1 = load_image(os.path.join(clip_dir, "im3.png"))

print("Input shape:", img0.shape)
print("GT shape:", gt.shape)

# RIFE middle-frame interpolation
with torch.no_grad():
    pred = model.inference(img0, img1)

print("Prediction shape:", pred.shape)
print("Prediction range:", pred.min().item(), "to", pred.max().item())

# Save prediction
pred_np = pred[0].clamp(0, 1).cpu().numpy()
pred_np = (pred_np.transpose(1, 2, 0) * 255).astype(np.uint8)

out_path = "/kaggle/working/rife_test_00058_0745.png"
cv2.imwrite(
    out_path,
    cv2.cvtColor(pred_np, cv2.COLOR_RGB2BGR)
)

print("Saved:", out_path)

PyTorch: 2.10.0+cu128
CUDA: True
RIFE root: /kaggle/input/datasets/jaisai002/rife-kaggle/RIFE_kaggle
RIFE model loaded successfully.
Input shape: torch.Size([1, 3, 256, 448])
GT shape: torch.Size([1, 3, 256, 448])
Prediction shape: torch.Size([1, 3, 256, 448])
Prediction range: 0.0 to 1.0
Saved: /kaggle/working/rife_test_00058_0745.png


In [7]:
import os
import pandas as pd

VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
RIFE_ROOT = "/kaggle/input/datasets/jaisai002/rife-kaggle/RIFE_kaggle"
SPLITS_PATH = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"

splits = pd.read_csv(SPLITS_PATH)

test_ids = splits.loc[splits["split"] == "test", "clip_id"].tolist()

print("Number of test clips:", len(test_ids))

test_clip = test_ids[0]
clip_dir = os.path.join(VIMEO_ROOT, "sequences", test_clip)

print("Test clip:", test_clip)
print("Files:", os.listdir(clip_dir))

Number of test clips: 3601
Test clip: 00001/0389
Files: ['im3.png', 'im1.png', 'im2.png']


In [8]:
import importlib.util

print("torchmetrics:", importlib.util.find_spec("torchmetrics"))
print("lpips:", importlib.util.find_spec("lpips"))
print("skimage:", importlib.util.find_spec("skimage"))

torchmetrics: ModuleSpec(name='torchmetrics', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7c3a58555070>, origin='/usr/local/lib/python3.12/dist-packages/torchmetrics/__init__.py', submodule_search_locations=['/usr/local/lib/python3.12/dist-packages/torchmetrics'])
lpips: None
skimage: ModuleSpec(name='skimage', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7c3a58c086e0>, origin='/usr/local/lib/python3.12/dist-packages/skimage/__init__.py', submodule_search_locations=['/usr/local/lib/python3.12/dist-packages/skimage'])


In [9]:
!pip install -q lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.4 MB/s eta 0:00:00


In [40]:
import lpips
import skimage
import torch

print("LPIPS:", lpips.__version__ if hasattr(lpips, "__version__") else "installed")
print("scikit-image:", skimage.__version__)
print("CUDA:", torch.cuda.is_available())

LPIPS: installed
scikit-image: 0.25.2
CUDA: True


In [10]:
import os
import sys
import time
import torch
import numpy as np
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import lpips

# ============================================================
# PATHS
# ============================================================
VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
RIFE_ROOT = "/kaggle/input/datasets/jaisai002/rife-kaggle/RIFE_kaggle"

clip_id = "00001/0389"
clip_dir = os.path.join(VIMEO_ROOT, "sequences", clip_id)

# Make RIFE importable
sys.path.insert(0, RIFE_ROOT)

from model.RIFE import Model

# ============================================================
# LOAD IMAGES
# ============================================================
def load_image(path):
    img = Image.open(path).convert("RGB")
    arr = np.array(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
    return tensor.cuda()

img0 = load_image(os.path.join(clip_dir, "im1.png"))
gt_np = np.array(
    Image.open(os.path.join(clip_dir, "im2.png")).convert("RGB")
).astype(np.float32) / 255.0

img1 = load_image(os.path.join(clip_dir, "im3.png"))

print("Input shape:", img0.shape)
print("GT shape:", gt_np.shape)

# ============================================================
# LOAD RIFE
# ============================================================
model = Model()
model.load_model(
    os.path.join(RIFE_ROOT, "train_log"),
    -1
)
model.eval()

print("RIFE loaded.")

# ============================================================
# WARM-UP
# ============================================================
with torch.no_grad():
    _ = model.inference(img0, img1)

torch.cuda.synchronize()

# ============================================================
# ACTUAL INFERENCE
# ============================================================
start = time.perf_counter()

with torch.no_grad():
    pred_tensor = model.inference(img0, img1)

torch.cuda.synchronize()
inference_time = time.perf_counter() - start

# ============================================================
# CONVERT PREDICTION TO NUMPY
# ============================================================
pred_tensor = pred_tensor.clamp(0, 1)

pred_np = (
    pred_tensor[0]
    .detach()
    .cpu()
    .permute(1, 2, 0)
    .numpy()
)

print("Prediction shape:", pred_np.shape)
print("Prediction range:", pred_np.min(), "to", pred_np.max())
print("Inference time:", inference_time, "seconds")

# ============================================================
# PSNR
# ============================================================
psnr = peak_signal_noise_ratio(
    gt_np,
    pred_np,
    data_range=1.0
)

# ============================================================
# SSIM
# ============================================================
ssim = structural_similarity(
    gt_np,
    pred_np,
    data_range=1.0,
    channel_axis=-1
)

# ============================================================
# LPIPS
# ============================================================
lpips_model = lpips.LPIPS(net="alex").cuda().eval()

pred_lpips = pred_tensor
gt_tensor = torch.from_numpy(gt_np).permute(2, 0, 1).unsqueeze(0).cuda()

# LPIPS expects [-1, 1]
pred_lpips = pred_lpips * 2.0 - 1.0
gt_tensor = gt_tensor * 2.0 - 1.0

with torch.no_grad():
    lpips_value = lpips_model(pred_lpips, gt_tensor).item()

# ============================================================
# SAVE PREDICTION
# ============================================================
pred_uint8 = (pred_np * 255.0).round().astype(np.uint8)

output_path = "/kaggle/working/rife_smoke_00001_0389.png"
Image.fromarray(pred_uint8).save(output_path)

# ============================================================
# RESULTS
# ============================================================
print("\n" + "=" * 50)
print("RIFE METRIC SMOKE TEST")
print("=" * 50)
print("Clip :", clip_id)
print(f"PSNR : {psnr:.4f} dB")
print(f"SSIM : {ssim:.6f}")
print(f"LPIPS: {lpips_value:.6f}")
print(f"Time : {inference_time:.4f} sec")
print("Saved:", output_path)

Input shape: torch.Size([1, 3, 256, 448])
GT shape: (256, 448, 3)
RIFE loaded.
Prediction shape: (256, 448, 3)
Prediction range: 0.0 to 0.90726376
Inference time: 0.02021685800002615 seconds
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 217MB/s]  


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth

RIFE METRIC SMOKE TEST
Clip : 00001/0389
PSNR : 38.9465 dB
SSIM : 0.980258
LPIPS: 0.015057
Time : 0.0202 sec
Saved: /kaggle/working/rife_smoke_00001_0389.png


#  **BATCH PROCESSING**

In [16]:
# import os
# import time
# import json
# import logging
# import torch
# import numpy as np
# import pandas as pd
# from PIL import Image
# from skimage.metrics import peak_signal_noise_ratio, structural_similarity
# from torch.utils.data import Dataset, DataLoader
# from tqdm.auto import tqdm
# import lpips as lpips_lib

# # ============================================================
# # PATHS
# # ============================================================
# VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
# RIFE_ROOT = "/kaggle/input/datasets/jaisai002/rife-kaggle/RIFE_kaggle"
# SPLITS_PATH = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"

# CATALOG_DIR = "/kaggle/working/catalog"
# LOG_DIR = "/kaggle/working/logs"
# os.makedirs(CATALOG_DIR, exist_ok=True)
# os.makedirs(LOG_DIR, exist_ok=True)

# OUTPUT_PATH = f"{CATALOG_DIR}/rife_metrics.csv"

# # ============================================================
# # LOGGING (file + console, resume-safe)
# # ============================================================
# logger = logging.getLogger("rife_batch")
# logger.setLevel(logging.INFO)
# logger.handlers.clear()

# fh = logging.FileHandler(f"{LOG_DIR}/rife_batch_run.log")
# fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
# ch = logging.StreamHandler()
# ch.setFormatter(logging.Formatter("%(message)s"))
# logger.addHandler(fh)
# logger.addHandler(ch)

# # ============================================================
# # TEST SPLIT (shared across all 4 notebooks — never re-derived)
# # ============================================================
# splits = pd.read_csv(SPLITS_PATH)
# test_ids = splits.loc[splits["split"] == "test", "clip_id"].tolist()
# logger.info(f"Test clips to process: {len(test_ids)}")

# # ============================================================
# # MODEL (loaded once)
# # ============================================================
# import sys
# if RIFE_ROOT not in sys.path:
#     sys.path.insert(0, RIFE_ROOT)
# from model.RIFE import Model

# model = Model()
# model.load_model(os.path.join(RIFE_ROOT, "train_log"), -1)
# model.eval()
# model.device()
# logger.info("RIFE model loaded.")

# lpips_model = lpips_lib.LPIPS(net="alex").cuda().eval()
# logger.info("LPIPS model loaded.")

# # ============================================================
# # DATASET (all Vimeo triplets are 256x448 -> safe to batch directly,
# # no resolution bucketing needed here unlike the RAFT notebook)
# # ============================================================
# class VimeoTripletDataset(Dataset):
#     def __init__(self, clip_ids):
#         self.clip_ids = clip_ids

#     def __len__(self):
#         return len(self.clip_ids)

#     def __getitem__(self, idx):
#         clip_id = self.clip_ids[idx]
#         clip_dir = os.path.join(VIMEO_ROOT, "sequences", clip_id)

#         im0 = np.array(Image.open(f"{clip_dir}/im1.png").convert("RGB")).astype(np.float32) / 255.0
#         gt = np.array(Image.open(f"{clip_dir}/im2.png").convert("RGB")).astype(np.float32) / 255.0
#         im1 = np.array(Image.open(f"{clip_dir}/im3.png").convert("RGB")).astype(np.float32) / 255.0

#         t0 = torch.from_numpy(im0).permute(2, 0, 1)
#         t1 = torch.from_numpy(im1).permute(2, 0, 1)
#         tgt = torch.from_numpy(gt).permute(2, 0, 1)

#         return clip_id, t0, t1, tgt

# # ============================================================
# # RESUME SUPPORT
# # ============================================================
# if os.path.exists(OUTPUT_PATH):
#     done_df = pd.read_csv(OUTPUT_PATH)

#     done_ids = set(
#         done_df.loc[
#             done_df["status"] == "success",
#             "clip_id"
#         ]
#     )

#     # Keep only successful results.
#     # Failed clips will be retried.
#     results = done_df[
#         done_df["status"] == "success"
#     ].to_dict("records")

#     logger.info(f"Resuming: {len(done_ids)} successful clips already done.")
# else:
#     done_ids = set()
#     results = []

# remaining_ids = [
#     c for c in test_ids
#     if c not in done_ids
# ]

# logger.info(f"Remaining to process: {len(remaining_ids)}")

# # ============================================================
# # BATCH LOOP
# # ============================================================
# BATCH_SIZE = 4
# NUM_WORKERS = 2
# CHECKPOINT_EVERY_N_BATCHES = 5

# ds = VimeoTripletDataset(remaining_ids)

# dl = DataLoader(
#     ds,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=NUM_WORKERS,
#     pin_memory=True
# )

# pbar = tqdm(
#     total=len(test_ids),
#     initial=len(done_ids),
#     unit="clip",
#     desc="RIFE eval"
# )

# processed = len(done_ids)
# batch_start = time.time()

# for batch_num, (clip_ids, im0_b, im1_b, gt_b) in enumerate(dl, start=1):

#     try:
#         im0_b = im0_b.cuda(non_blocking=True)
#         im1_b = im1_b.cuda(non_blocking=True)

#         # --------------------------------------------
#         # Accurate GPU inference timing
#         # --------------------------------------------
#         torch.cuda.synchronize()
#         t0 = time.perf_counter()

#         with torch.no_grad():
#             pred_b = model.inference(im0_b, im1_b)

#         torch.cuda.synchronize()
#         elapsed_batch = time.perf_counter() - t0

#         pred_b = pred_b.clamp(0, 1)

#         per_clip_time = elapsed_batch / len(clip_ids)

#         # --------------------------------------------
#         # LPIPS
#         # --------------------------------------------
#         pred_lpips_in = pred_b * 2.0 - 1.0
#         gt_lpips_in = gt_b.cuda(non_blocking=True) * 2.0 - 1.0

#         with torch.no_grad():
#             lpips_batch_vals = (
#                 lpips_model(
#                     pred_lpips_in,
#                     gt_lpips_in
#                 )
#                 .flatten()
#                 .detach()
#                 .cpu()
#                 .numpy()
#             )

#         # --------------------------------------------
#         # Convert to CPU for PSNR / SSIM
#         # --------------------------------------------
#         pred_np_batch = (
#             pred_b
#             .detach()
#             .cpu()
#             .permute(0, 2, 3, 1)
#             .numpy()
#         )

#         gt_np_batch = (
#             gt_b
#             .permute(0, 2, 3, 1)
#             .numpy()
#         )

#         # --------------------------------------------
#         # Metrics per clip
#         # --------------------------------------------
#         for i, cid in enumerate(clip_ids):

#             psnr = peak_signal_noise_ratio(
#                 gt_np_batch[i],
#                 pred_np_batch[i],
#                 data_range=1.0
#             )

#             ssim = structural_similarity(
#                 gt_np_batch[i],
#                 pred_np_batch[i],
#                 data_range=1.0,
#                 channel_axis=-1
#             )

#             results.append({
#                 "clip_id": cid,
#                 "psnr": float(psnr),
#                 "ssim": float(ssim),
#                 "lpips": float(lpips_batch_vals[i]),
#                 "inference_time_sec": float(per_clip_time),
#                 "status": "success"
#             })

#         processed += len(clip_ids)
#         pbar.update(len(clip_ids))

#         # --------------------------------------------
#         # Checkpoint
#         # --------------------------------------------
#         if batch_num % CHECKPOINT_EVERY_N_BATCHES == 0:

#             pd.DataFrame(results).to_csv(
#                 OUTPUT_PATH,
#                 index=False
#             )

#             elapsed = time.time() - batch_start

#             rate = (
#                 (processed - len(done_ids)) / elapsed
#                 if elapsed > 0 else 0
#             )

#             eta = (
#                 (len(test_ids) - processed) / rate
#                 if rate > 0 else float("nan")
#             )

#             pbar.set_postfix_str(
#                 f"rate={rate:.2f} clip/s "
#                 f"ETA={eta/60:.1f}min"
#             )

#             logger.info(
#                 f"[{processed}/{len(test_ids)}] "
#                 f"elapsed={elapsed/60:.1f}min "
#                 f"rate={rate:.2f} clips/s "
#                 f"ETA={eta/60:.1f}min"
#             )

#     except Exception as e:

#         logger.error(
#             f"BATCH FAILED "
#             f"(batch {batch_num}): {repr(e)}"
#         )

#         for cid in clip_ids:
#             results.append({
#                 "clip_id": cid,
#                 "psnr": np.nan,
#                 "ssim": np.nan,
#                 "lpips": np.nan,
#                 "inference_time_sec": np.nan,
#                 "status": "failed"
#             })

#         processed += len(clip_ids)
#         pbar.update(len(clip_ids))

# pbar.close()

# # final flush
# final_df = pd.DataFrame(results)
# final_df.to_csv(OUTPUT_PATH, index=False)

# logger.info("========================================")
# logger.info("RIFE BATCH COMPLETE")
# logger.info("========================================")
# logger.info(f"Successful: {(final_df['status'] == 'success').sum()}")
# logger.info(f"Failed:     {(final_df['status'] == 'failed').sum()}")
# logger.info(f"Mean PSNR:  {final_df['psnr'].mean():.4f} dB")
# logger.info(f"Mean SSIM:  {final_df['ssim'].mean():.6f}")
# logger.info(f"Mean LPIPS: {final_df['lpips'].mean():.6f}")
# logger.info(f"Total time: {(time.time() - batch_start)/60:.2f} minutes")
# logger.info(f"Saved to:   {OUTPUT_PATH}")

# # ============================================================
# # MANIFEST (for join with RAFT categories + other VFI models later)
# # ============================================================
# manifest = {
#     "model": "RIFE",
#     "checkpoint": "RIFE_kaggle/train_log",
#     "split_source": SPLITS_PATH,
#     "config": {"batch_size": BATCH_SIZE},
#     "counts": {
#         "total": int(len(final_df)),
#         "success": int((final_df["status"] == "success").sum()),
#         "failed": int((final_df["status"] == "failed").sum())
#     },
#     "files": {
#         "metrics": OUTPUT_PATH,
#         "log": f"{LOG_DIR}/rife_batch_run.log"
#     }
# }
# with open(f"{CATALOG_DIR}/rife_manifest.json", "w") as f:
#     json.dump(manifest, f, indent=2)
# logger.info(f"Saved manifest -> {CATALOG_DIR}/rife_manifest.json")


In [11]:
# ============================================================
# RIFE CORRECTION CNN
# 2-GPU TRAINING SETUP
# ============================================================

import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

from PIL import Image
from torch.utils.data import Dataset, DataLoader

print("GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
CLASSIFIED_CSV = "/kaggle/input/datasets/hibro00/vfi-csvs/raft_descriptors_train_classified.csv"

OUTPUT_DIR = "/kaggle/working/rife_cnn"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GPUs: 2
0 Tesla T4
1 Tesla T4


In [12]:
# ============================================================
# LOAD CLASSIFIED TRAINING DATA
# ============================================================

train_df = pd.read_csv(CLASSIFIED_CSV)

train_df = train_df[
    train_df["status"] == "success"
].copy()

rotation_df = train_df[
    train_df["predicted_category"] == "rotation"
].copy()

print("Generic training clips:", len(train_df))
print("Targeted rotation clips:", len(rotation_df))

Generic training clips: 51324
Targeted rotation clips: 2980


In [13]:
# ============================================================
# VIMEO DATASET
# ============================================================

class CorrectionDataset(Dataset):

    def __init__(self, dataframe):
        self.clip_ids = dataframe["clip_id"].tolist()

    def __len__(self):
        return len(self.clip_ids)

    def __getitem__(self, idx):

        clip_id = self.clip_ids[idx]

        clip_dir = os.path.join(
            VIMEO_ROOT,
            "sequences",
            clip_id
        )

        I0 = np.array(
            Image.open(
                os.path.join(clip_dir, "im1.png")
            ).convert("RGB"),
            dtype=np.float32
        ) / 255.0

        GT = np.array(
            Image.open(
                os.path.join(clip_dir, "im2.png")
            ).convert("RGB"),
            dtype=np.float32
        ) / 255.0

        I1 = np.array(
            Image.open(
                os.path.join(clip_dir, "im3.png")
            ).convert("RGB"),
            dtype=np.float32
        ) / 255.0

        I0 = torch.from_numpy(I0).permute(2, 0, 1)
        GT = torch.from_numpy(GT).permute(2, 0, 1)
        I1 = torch.from_numpy(I1).permute(2, 0, 1)

        return I0, I1, GT, clip_id

In [15]:
# ============================================================
# LIGHTWEIGHT CORRECTION CNN
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1)
        )

    def forward(self, x):
        return x + self.block(x)


class CorrectionCNN(nn.Module):

    def __init__(self, base_channels=32):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(9, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),

            ResidualBlock(base_channels),
            ResidualBlock(base_channels),
            ResidualBlock(base_channels),

            nn.Conv2d(
                base_channels,
                base_channels,
                3,
                padding=1
            ),
            nn.ReLU(inplace=True)
        )

        # 3-channel residual
        self.residual_head = nn.Conv2d(
            base_channels,
            3,
            3,
            padding=1
        )

        # 1-channel confidence mask
        self.mask_head = nn.Sequential(
            nn.Conv2d(
                base_channels,
                1,
                3,
                padding=1
            ),
            nn.Sigmoid()
        )

    def forward(self, I0, I1, Ib):

        x = torch.cat(
            [I0, I1, Ib],
            dim=1
        )

        features = self.encoder(x)

        residual = self.residual_head(features)
        mask = self.mask_head(features)

        corrected = Ib + mask * residual

        corrected = torch.clamp(
            corrected,
            0.0,
            1.0
        )

        return corrected, residual, mask

In [16]:
# ============================================================
# FREEZE RIFE
# RIFE Model wrapper is not an nn.Module.
# Its inference is already run under torch.no_grad().
# ============================================================

model.eval()

print("RIFE loaded and set to evaluation mode.")
print("RIFE parameters are excluded from CNN gradients via torch.no_grad().")

RIFE loaded and set to evaluation mode.
RIFE parameters are excluded from CNN gradients via torch.no_grad().


In [17]:
# ============================================================
# 2-GPU CORRECTION CNN
# ============================================================

assert torch.cuda.device_count() >= 2, \
    "At least 2 GPUs are required."

device = torch.device("cuda:0")

correction_model = CorrectionCNN(
    base_channels=32
)

correction_model = nn.DataParallel(
    correction_model,
    device_ids=[0, 1]
)

correction_model = correction_model.to(device)

print("Correction CNN GPUs:", correction_model.device_ids)

Correction CNN GPUs: [0, 1]


In [18]:
# ============================================================
# LOSS
# ============================================================

def charbonnier_loss(pred, target, eps=1e-3):
    diff = pred - target
    return torch.mean(
        torch.sqrt(diff * diff + eps * eps)
    )


def correction_loss(
    corrected,
    target,
    residual
):

    l1 = F.l1_loss(
        corrected,
        target
    )

    charbonnier = charbonnier_loss(
        corrected,
        target
    )

    residual_reg = torch.mean(
        torch.abs(residual)
    )

    loss = (
        l1
        + 0.1 * charbonnier
        + 0.001 * residual_reg
    )

    return loss

In [19]:
# ============================================================
# FAULT-TOLERANT TRAINING WITH EPOCH CHECKPOINTS
# ============================================================

from tqdm.auto import tqdm


def _strip_module(state_dict):
    """Remove the DataParallel 'module.' prefix if present."""
    return {
        (k[len("module."):] if k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }


def train_correction(
    dataframe,
    save_name,
    epochs=10,
    batch_size=4,
    lr=1e-4,
    init_weights=None,   # optional: weights-only file to start from
):
    dataset = CorrectionDataset(dataframe)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        drop_last=True,
    )

    optimizer = torch.optim.Adam(correction_model.parameters(), lr=lr)

    # FIX 1: checkpoint and final model always get different paths,
    # whether or not save_name ends with ".pth"
    base = save_name[:-4] if save_name.endswith(".pth") else save_name
    checkpoint_path = os.path.join(OUTPUT_DIR, f"{base}_checkpoint.pth")
    final_path = os.path.join(OUTPUT_DIR, f"{base}.pth")

    start_epoch = 0          # FIX 2: 0-indexed, so a fresh run is epoch 0
    best_loss = float("inf")
    history = []

    # ========================================================
    # RESUME
    # ========================================================
    if os.path.exists(checkpoint_path):
        print("Checkpoint found:", checkpoint_path)
        ckpt = torch.load(checkpoint_path, map_location="cuda:0")

        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            correction_model.module.load_state_dict(
                _strip_module(ckpt["model_state_dict"])
            )
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            start_epoch = ckpt["epoch"] + 1
            best_loss = ckpt.get("best_loss", float("inf"))
            history = ckpt.get("history", [])
            print(f"Resuming from epoch {start_epoch + 1}/{epochs}")
        else:
            sd = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
            correction_model.module.load_state_dict(_strip_module(sd))
            print("Weights-only checkpoint: fresh optimizer, epoch 1.")

    elif init_weights is not None and os.path.exists(init_weights):
        sd = torch.load(init_weights, map_location="cuda:0")
        if isinstance(sd, dict) and "model_state_dict" in sd:
            sd = sd["model_state_dict"]
        correction_model.module.load_state_dict(_strip_module(sd))
        print("Initialised from weights:", init_weights)
        print("Fresh optimizer, epoch 1.")

    else:
        print("No checkpoint found. Starting from epoch 1.")

    # ========================================================
    # TRAIN
    # ========================================================
    print("\n" + "=" * 60)
    print("TRAINING:", save_name)
    print("Clips:", len(dataset))
    print("Epochs:", epochs, f"(running {max(0, epochs - start_epoch)})")
    print("Batch size:", batch_size)
    print("GPUs:", [0, 1])
    print("=" * 60)

    for epoch in range(start_epoch, epochs):

        correction_model.train()
        epoch_loss = 0.0
        n_ok = 0
        start_time = time.time()

        progress = tqdm(
            loader,
            total=len(loader),
            desc=f"Epoch {epoch + 1}/{epochs}",
            unit="batch",
        )

        for batch_num, (I0, I1, GT, clip_ids) in enumerate(progress):
            try:
                I0 = I0.cuda(0, non_blocking=True)
                I1 = I1.cuda(0, non_blocking=True)
                GT = GT.cuda(0, non_blocking=True)

                # Frozen RIFE
                with torch.no_grad():
                    Ib = model.inference(I0, I1)
                    Ib = torch.clamp(Ib, 0.0, 1.0)

                optimizer.zero_grad(set_to_none=True)

                corrected, residual, mask = correction_model(I0, I1, Ib)
                loss = correction_loss(corrected, GT, residual)

                if not torch.isfinite(loss):
                    print(f"\nWARNING: non-finite loss at batch {batch_num}")
                    optimizer.zero_grad(set_to_none=True)
                    continue

                loss.backward()
                optimizer.step()

                epoch_loss += loss.item()
                n_ok += 1
                progress.set_postfix(
                    loss=f"{loss.item():.5f}",
                    avg=f"{epoch_loss / n_ok:.5f}",
                )

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\nCUDA OOM at batch {batch_num}. Skipping batch.")
                    optimizer.zero_grad(set_to_none=True)
                    torch.cuda.empty_cache()
                    continue
                raise

        avg_loss = epoch_loss / max(1, n_ok)
        elapsed = time.time() - start_time
        history.append(avg_loss)

        print(
            f"\nEpoch {epoch + 1}/{epochs} complete | "
            f"Avg Loss: {avg_loss:.6f} | Time: {elapsed / 60:.2f} min"
        )

        # ====================================================
        # SAVE CHECKPOINT (full state, separate from final model)
        # ====================================================
        best_loss = min(best_loss, avg_loss)

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": correction_model.module.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": avg_loss,
                "best_loss": best_loss,
                "history": history,
                "epochs": epochs,
                "batch_size": batch_size,
                "learning_rate": lr,
            },
            checkpoint_path,
        )
        print("Checkpoint saved:", checkpoint_path)

    # ========================================================
    # FINAL MODEL (weights only, different file from checkpoint)
    # ========================================================
    torch.save(correction_model.module.state_dict(), final_path)

    print("\nTraining complete.")
    print("Final model:", final_path)

    # FIX 3: return both values so the caller can unpack them
    return final_path, history

In [20]:
import shutil
import os

src_final = "/kaggle/input/datasets/hibro00/vfi-pths-backup/rife_generic_correction.pth"
src_checkpoint = "/kaggle/input/datasets/hibro00/vfi-pths-backup/rife_generic_correction_checkpoint.pth"

dst_dir = "/kaggle/working/rife_cnn"

os.makedirs(dst_dir, exist_ok=True)

shutil.copy2(
    src_final,
    os.path.join(
        dst_dir,
        "rife_generic_correction.pth"
    )
)

shutil.copy2(
    src_checkpoint,
    os.path.join(
        dst_dir,
        "rife_generic_correction_checkpoint.pth"
    )
)

print("Final:", os.path.exists(
    os.path.join(dst_dir, "rife_generic_correction.pth")
))

print("Checkpoint:", os.path.exists(
    os.path.join(
        dst_dir,
        "rife_generic_correction_checkpoint.pth"
    )
))

Final: True
Checkpoint: True


In [60]:
# # ============================================================
# # CONDITION B
# # GENERIC CORRECTION CNN
# # ============================================================

# generic_checkpoint = train_correction(
#     dataframe=train_df,
#     save_name="rife_generic_correction.pth",
#     epochs=10,
#     batch_size=4,
#     lr=1e-4
# )

In [71]:
# EPOCHS = 5
# BATCH_SIZE = 4
# LR = 1e-4

# rotation_path, rotation_history = train_correction(
#     rotation_df,
#     "rife_correction_rotation",
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     lr=LR,
#     # Reuse the weights from your last run, which were saved without a .pth extension.
#     # Delete this line to train from scratch.
#     init_weights="/kaggle/working/rife_cnn/rife_correction_rotation",
# )

# print(rotation_path)
# print(rotation_history)

Initialised from weights: /kaggle/working/rife_cnn/rife_correction_rotation
Fresh optimizer, epoch 1.

TRAINING: rife_correction_rotation
Clips: 2980
Epochs: 5 (running 5)
Batch size: 4
GPUs: [0, 1]


Epoch 1/5:   0%|          | 0/745 [00:00<?, ?batch/s]


Epoch 1/5 complete | Avg Loss: 0.009653 | Time: 1.33 min
Checkpoint saved: /kaggle/working/rife_cnn/rife_correction_rotation_checkpoint.pth


Epoch 2/5:   0%|          | 0/745 [00:00<?, ?batch/s]


Epoch 2/5 complete | Avg Loss: 0.009646 | Time: 1.35 min
Checkpoint saved: /kaggle/working/rife_cnn/rife_correction_rotation_checkpoint.pth


...


Epoch 3/5 complete | Avg Loss: 0.009640 | Time: 1.35 min
Checkpoint saved: /kaggle/working/rife_cnn/rife_correction_rotation_checkpoint.pth


Epoch 4/5:   0%|          | 0/745 [00:00<?, ?batch/s]


Epoch 4/5 complete | Avg Loss: 0.009632 | Time: 1.34 min
Checkpoint saved: /kaggle/working/rife_cnn/rife_correction_rotation_checkpoint.pth


Epoch 5/5:   0%|          | 0/745 [00:00<?, ?batch/s]


Epoch 5/5 complete | Avg Loss: 0.009623 | Time: 1.34 min
Checkpoint saved: /kaggle/working/rife_cnn/rife_correction_rotation_checkpoint.pth

Training complete.
Final model: /kaggle/working/rife_cnn/rife_correction_rotation.pth
/kaggle/working/rife_cnn/rife_correction_rotation.pth
[0.009653312104735398, 0.00964607320188376, 0.009639931200759722, 0.0096321684114585, 0.009622553367505237]


# ***FINAL EVALUATION — BASELINE vs GENERIC vs ROTATION (test split)***


In [24]:
# ============================================================
# RIFE: BASELINE vs GENERIC CNN vs ROTATION CNN (test split)
# ============================================================
import lpips
from scipy import stats
from skimage.metrics import structural_similarity as sk_ssim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

GEN_W = os.path.join(OUTPUT_DIR, "rife_generic_correction.pth")
if not os.path.exists(GEN_W):
    GEN_W = "/kaggle/working/rife_generic_correction.pth"

ROT_W = os.path.join(OUTPUT_DIR, "rife_correction_rotation.pth")
if not os.path.exists(ROT_W):
    ROT_W = "/kaggle/working/rife_correction_rotation.pth"

OUT_CSV   = "/kaggle/working/rife_baseline_generic_rotation.csv"
EVAL_BS   = 4
MAX_CLIPS = None   # e.g. 64 for a quick trial run

for p in (GEN_W, ROT_W):
    assert os.path.exists(p), f"Missing weights: {p}"
print("Generic weights :", GEN_W)
print("Rotation weights:", ROT_W)

# ------------------------------------------------------------
# LOAD CNNs (handles full checkpoints and DataParallel prefixes)
# ------------------------------------------------------------
def load_cnn(path):
    sd = torch.load(path, map_location="cpu")
    if isinstance(sd, dict) and "model_state_dict" in sd:
        sd = sd["model_state_dict"]
    sd = {k[len("module."):] if k.startswith("module.") else k: v for k, v in sd.items()}
    net = CorrectionCNN(base_channels=32)
    net.load_state_dict(sd, strict=True)
    return net.to("cuda:0").eval()

gen_net = load_cnn(GEN_W)
rot_net = load_cnn(ROT_W)
lp = lpips.LPIPS(net="alex").to("cuda:0").eval()

# ------------------------------------------------------------
# TEST CLIPS
# ------------------------------------------------------------
sp = pd.read_csv(SPLITS_PATH)
sp["clip_id"] = sp["clip_id"].astype(str)
ids = list(dict.fromkeys(sp.loc[sp["split"].astype(str).str.lower() == "test", "clip_id"]))
if MAX_CLIPS:
    ids = ids[:MAX_CLIPS]
print("Test clips:", len(ids))

eval_loader = DataLoader(
    CorrectionDataset(pd.DataFrame({"clip_id": ids})),
    batch_size=EVAL_BS, shuffle=False, num_workers=2, pin_memory=True,
)

# ------------------------------------------------------------
# BATCH METRICS
# ------------------------------------------------------------
def psnr_b(a, b):
    mse = ((a - b) ** 2).flatten(1).mean(1).clamp_min(1e-10)
    return (10 * torch.log10(1.0 / mse)).cpu().numpy()

def ssim_b(a, b):
    a = a.cpu().numpy().transpose(0, 2, 3, 1)
    b = b.cpu().numpy().transpose(0, 2, 3, 1)
    return np.array([sk_ssim(b[i], a[i], channel_axis=2, data_range=1.0) for i in range(len(a))])

def lpips_b(a, b):
    return lp(a * 2 - 1, b * 2 - 1).flatten().cpu().numpy()

# ------------------------------------------------------------
# EVALUATION LOOP
# ------------------------------------------------------------
rows = []
with torch.no_grad():
    for bi, (I0, I1, GT, cids) in enumerate(tqdm(eval_loader, desc="RIFE eval")):
        I0, I1, GT = I0.cuda(0), I1.cuda(0), GT.cuda(0)
        ib = model.inference(I0, I1).clamp(0, 1)      # frozen RIFE baseline

        outs = {
            "base": ib,
            "gen":  gen_net(I0, I1, ib)[0].clamp(0, 1),
            "rot":  rot_net(I0, I1, ib)[0].clamp(0, 1),
        }
        m = {k: (psnr_b(v, GT), ssim_b(v, GT), lpips_b(v, GT)) for k, v in outs.items()}

        for j, cid in enumerate(cids):
            r = {"clip_id": cid}
            for k in outs:
                r[f"{k}_psnr"]  = float(m[k][0][j])
                r[f"{k}_ssim"]  = float(m[k][1][j])
                r[f"{k}_lpips"] = float(m[k][2][j])
            rows.append(r)

        if (bi + 1) % 50 == 0:
            pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

res = pd.DataFrame(rows)
assert len(res) == len(ids), f"Expected {len(ids)} rows, got {len(res)}"

# ------------------------------------------------------------
# CATEGORIES + DELTAS
# ------------------------------------------------------------
cat = pd.read_csv(TEST_CLASSIFIED)
cat["clip_id"] = cat["clip_id"].astype(str)
res = res.merge(cat[["clip_id", "predicted_category"]].drop_duplicates("clip_id"),
                on="clip_id", how="left").rename(columns={"predicted_category": "category"})
res["category"] = res["category"].fillna("unknown")

for model_key in ("gen", "rot"):
    for met in ("psnr", "ssim", "lpips"):
        res[f"{model_key}_d_{met}"] = res[f"{model_key}_{met}"] - res[f"base_{met}"]

res.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV, "| clips:", len(res))

# ------------------------------------------------------------
# SUMMARY BY CATEGORY
# ------------------------------------------------------------
def summarize(g):
    return pd.Series({
        "n": len(g),
        "base_psnr": g["base_psnr"].mean(),
        "gen_d_psnr": g["gen_d_psnr"].mean(),
        "gen_d_ssim": g["gen_d_ssim"].mean(),
        "gen_d_lpips": g["gen_d_lpips"].mean(),
        "gen_win_%": 100 * (g["gen_d_psnr"] > 0).mean(),
        "rot_d_psnr": g["rot_d_psnr"].mean(),
        "rot_d_ssim": g["rot_d_ssim"].mean(),
        "rot_d_lpips": g["rot_d_lpips"].mean(),
        "rot_win_%": 100 * (g["rot_d_psnr"] > 0).mean(),
    })

by_cat  = res.groupby("category").apply(summarize).reset_index()
overall = summarize(res).to_frame().T
overall.insert(0, "category", "ALL")
final = pd.concat([by_cat, overall], ignore_index=True)

print("\nd_* = corrected minus baseline. PSNR/SSIM: higher is better. LPIPS: lower is better.\n")
display(final.round(4))

# ------------------------------------------------------------
# SIGNIFICANCE (paired t-test vs 0)
# ------------------------------------------------------------
for model_key, label in (("gen", "Generic"), ("rot", "Rotation")):
    d = res[f"{model_key}_d_psnr"]
    print(f"\n{label} PSNR delta: mean {d.mean():+.4f} dB | median {d.median():+.4f} | "
          f"p = {stats.ttest_1samp(d, 0).pvalue:.3g}")


Generic weights : /kaggle/working/rife_cnn/rife_generic_correction.pth
Rotation weights: /kaggle/working/rife_cnn/rife_correction_rotation.pth
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Test clips: 3601


RIFE eval:   0%|          | 0/901 [00:00<?, ?it/s]

Saved: /kaggle/working/rife_baseline_generic_rotation.csv | clips: 3601

d_* = corrected minus baseline. PSNR/SSIM: higher is better. LPIPS: lower is better.



/tmp/ipykernel_58/2074103818.py:136: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_cat  = res.groupby("category").apply(summarize).reset_index()


,category,n,base_psnr,gen_d_psnr,gen_d_ssim,gen_d_lpips,gen_win_%,rot_d_psnr,rot_d_ssim,rot_d_lpips,rot_win_%
0,near_linear,1928.0,36.5787,0.1757,0.0013,0.0000,93.7759,0.0351,0.0002,-0.0001,64.8340
1,non_rigid,1171.0,34.7366,0.1072,0.0012,-0.0001,93.9368,0.0306,0.0002,-0.0002,67.8907
2,occlusion,322.0,34.9091,0.1035,0.0012,-0.0001,94.7205,0.0201,0.0002,-0.0001,68.3230
3,rotation,180.0,32.8763,0.0907,0.0012,-0.0003,94.4444,0.0325,0.0003,-0.0002,79.4444
4,ALL,3601.0,35.6453,0.1427,0.0013,-0.0000,93.9461,0.0321,0.0002,-0.0002,66.8703



Generic PSNR delta: mean +0.1427 dB | median +0.0781 | p = 1.15e-292

Rotation PSNR delta: mean +0.0321 dB | median +0.0174 | p = 2.82e-96
